In [8]:
#pull processed data
import pandas as pd
from pathlib import Path

dia_df = pd.read_csv("../data/processed/flight_log.csv")
print(dia_df.head(2))

  Carrier Code  Flight Number  Scheduled elapsed time (Minutes)  \
0           WN          540.0                             125.0   
1           WN         2751.0                             150.0   

   Actual elapsed time (Minutes)  Departure delay (Minutes)  \
0                          125.0                        1.0   
1                          152.0                        0.0   

   Taxi-Out time (Minutes)  Delay Carrier (Minutes)  Delay Weather (Minutes)  \
0                     25.0                      0.0                      0.0   
1                     18.0                      0.0                      0.0   

   Delay National Aviation System (Minutes)  Delay Security (Minutes)  ...  \
0                                       0.0                       0.0  ...   
1                                       0.0                       0.0  ...   

   monthly_passenger_arr monthly_freight_arr total_arr  monthly_seats_arr  \
0                2552684            25223364     23321 

In [9]:
#add column for delay > 15 minutes
dia_df["15min_delay"] = (dia_df["Departure delay (Minutes)"] > 15).astype(int)
dia_df["Datetime Departure"] = pd.to_datetime(dia_df["Datetime Departure"])
dia_df["Datetime Arrival"] = pd.to_datetime(dia_df["Datetime Arrival"])

#rolling averages based off departure time
dia_df.sort_values(by = "Datetime Departure", inplace=True)
dia_df = dia_df.set_index("Datetime Departure")
#rolling mean on departure delay
dia_df["dep_delay_mean3h"] = dia_df["Departure delay (Minutes)"].rolling("3h", min_periods=1).mean()
dia_df["dep_delay_mean6h"] = dia_df["Departure delay (Minutes)"].rolling("6h", min_periods=1).mean()
dia_df["dep_delay_mean12h"] = dia_df["Departure delay (Minutes)"].rolling("12h", min_periods=1).mean()
#rolling 3 hour delay count
dia_df["dep_delay_count3h"] = dia_df["15min_delay"].rolling("3h", min_periods=1).sum()
dia_df["dep_delay_count6h"] = dia_df["15min_delay"].rolling("6h", min_periods=1).sum()
dia_df["dep_delay_count12h"] = dia_df["15min_delay"].rolling("12h", min_periods=1).sum()
#number of scheduled flights
dia_df["dep_last_1h"] = dia_df["Flight Number"].rolling("1h", min_periods=1).count()
dia_df["dep_last_3h"] = dia_df["Flight Number"].rolling("3h", min_periods=1).count()
#bank density
dia_df["bank_density_15min"] = dia_df["Flight Number"].rolling("30min", center=True, min_periods=1).count()

dia_df = dia_df.reset_index()

#turnaround time in between arrival and delay
dia_df["turnaround_slack"] = (
    dia_df["Datetime Departure"] - dia_df["Datetime Arrival"]
).dt.total_seconds() / 60

dia_df.sort_values("Datetime Departure")
dia_df.drop(columns=['Datetime Departure', 'Datetime Arrival','Flight Number', 'Carrier Code'])



,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Taxi-Out time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes),Arrival Delay (Minutes),...,dep_delay_mean3h,dep_delay_mean6h,dep_delay_mean12h,dep_delay_count3h,dep_delay_count6h,dep_delay_count12h,dep_last_1h,dep_last_3h,bank_density_15min,turnaround_slack
0,125.0,125.0,1.0,25.0,0.0,0.0,0.0,0.0,0.0,-12.0,...,1.000000,1.000000,1.000000,0.0,0.0,0.0,1.0,1.0,1.0,272.0
1,150.0,152.0,0.0,18.0,0.0,0.0,0.0,0.0,0.0,-11.0,...,0.500000,0.500000,0.500000,0.0,0.0,0.0,2.0,2.0,1.0,326.0
2,110.0,129.0,-13.0,28.0,0.0,0.0,0.0,0.0,0.0,36.0,...,-4.000000,-4.000000,-4.000000,0.0,0.0,0.0,3.0,3.0,2.0,327.0
3,128.0,121.0,-9.0,16.0,0.0,0.0,0.0,0.0,0.0,74.0,...,-5.250000,-5.250000,-5.250000,0.0,0.0,0.0,3.0,4.0,2.0,267.0
4,125.0,116.0,0.0,19.0,0.0,0.0,0.0,0.0,0.0,-5.0,...,-4.200000,-4.200000,-4.200000,0.0,0.0,0.0,4.0,5.0,1.0,45.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
442159,223.0,216.0,-2.0,43.0,0.0,0.0,0.0,0.0,0.0,-16.0,...,12.611111,9.849398,8.025714,11.0,35.0,60.0,9.0,36.0,2.0,113.0
442160,204.0,228.0,-10.0,30.0,0.0,0.0,0.0,0.0,0.0,36.0,...,12.153846,8.838028,8.163009,8.0,27.0,58.0,4.0,26.0,4.0,118.0
442161,198.0,193.0,22.0,35.0,17.0,0.0,0.0,0.0,0.0,0.0,...,11.900000,8.094488,8.206250,6.0,24.0,59.0,5.0,20.0,4.0,120.0
442162,182.0,168.0,-5.0,32.0,0.0,0.0,0.0,0.0,0.0,152.0,...,10.058824,9.163793,8.165109,4.0,24.0,59.0,5.0,17.0,4.0,280.0


In [10]:
dia_df.drop(columns=['Flight Number','Datetime Departure','Datetime Arrival','Carrier Code','Delay Carrier (Minutes)',
                     'Delay Weather (Minutes)','Delay National Aviation System (Minutes)','Delay Late Aircraft Arrival (Minutes)',
                     'Scheduled elapsed time (Minutes)','Actual elapsed time (Minutes)','Taxi-Out time (Minutes)','Delay Security (Minutes)' ], 
                     inplace=True)
print(dia_df.info())

filepath = Path("../data/processed")
dia_df.to_csv(filepath /"dia_flights.csv", index= False)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442164 entries, 0 to 442163
Data columns (total 30 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Departure delay (Minutes)  442164 non-null  float64
 1   Arrival Delay (Minutes)    442164 non-null  float64
 2   temp                       442164 non-null  float64
 3   dwpt                       442164 non-null  float64
 4   rhum                       442164 non-null  float64
 5   prcp                       442164 non-null  float64
 6   wdir                       442164 non-null  float64
 7   wspd                       442164 non-null  float64
 8   pres                       442164 non-null  float64
 9   monthly_passenger_arr      442164 non-null  int64  
 10  monthly_freight_arr        442164 non-null  int64  
 11  total_arr                  442164 non-null  int64  
 12  monthly_seats_arr          442164 non-null  int64  
 13  monthly_passenger_dep      44